# Retrieval
---

## Información


### Objetivo

Implementar la recuperación semántica de información utilizando los embeddings almacenados en ChromaDB, obteniendo los fragmentos más relevantes para una consulta del usuario.

### Objetivos específicos
* Cargar el modelo de embeddings utilizado durante la indexación.
* Conectar con el vector store persistente.
* Realizar consultas semánticas.
* Recuperar fragmentos relevantes del documento.
* Validar que los resultados obtenidos sean coherentes con la consulta.

### Entradas
* Vector store generado en la etapa de embeddings.
* Modelo sentence-transformers/all-MiniLM-L6-v2.

### Salidas
* Lista de fragmentos relevantes recuperados.
* Contexto inicial para la etapa de generación.

---
## Etapa 1. Carga del vector store

En esta etapa se carga la colección persistente de ChromaDB creada durante la etapa de Embeddings.

El objetivo es reutilizar el índice vectorial previamente generado para realizar búsquedas semánticas sin necesidad de volver a calcular los embeddings de los documentos.

In [37]:
from pathlib import Path
import sys

# Agregar la raíz del proyecto al PATH para importar módulos de src
project_root = Path.cwd().parent
sys.path.append(str(project_root))

vector_store_path = project_root / "data" / "vector_store"

### Importar módulos de Vector Store

In [38]:
import src.processing.vector_store as vector_store
from src.processing.vector_store import get_or_create_collection

### Cargar la colección existente

In [39]:
collection = get_or_create_collection(
    collection_name="alessia_collection",
    vector_store_path=vector_store_path
)

### Validación de la colección

In [40]:
print(f"Colección: {collection.name}")
print(f"Cantidad de vectores: {collection.count()}")

Colección: alessia_collection
Cantidad de vectores: 229


## Etapa 2. Cargar el modelo de embeddings

### 1. Importar SentenceTransformer

In [41]:
from sentence_transformers import SentenceTransformer

### 2. Cargar el modelo

In [42]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4015.70it/s]


## Etapa 3. Recuperación semántica
En esta etapa se realiza una consulta al vector store utilizando una pregunta del usuario.
El texto de la consulta es transformado en un embedding utilizando el mismo modelo empleado durante la indexación.
ChromaDB compara el embedding generado con los vectores almacenados y devuelve los fragmentos con mayor similitud semántica.

In [43]:
from src.processing.vector_store import search_chunks

In [44]:
query = """
¿Qué medidas debe tomar la comunidad ante una emergencia por inundación?
"""

In [45]:
retrieval_results = search_chunks(
    query=query,
    collection=collection,
    model=model,
    k=5,
)

In [46]:
print(retrieval_results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


### Mostrar los fragmentos recuperados

In [47]:
documents = retrieval_results["documents"][0]
metadatas = retrieval_results["metadatas"][0]
distances = retrieval_results["distances"][0]

for i, (document, metadata, distance) in enumerate(
    zip(documents, metadatas, distances),
    start=1,
):
    print("=" * 80)
    print(f"Resultado {i}")
    print(f"Distancia : {distance:.4f}")
    print(f"Metadata  : {metadata}")
    print()
    print(document)
    print()

Resultado 1
Distancia : 0.5823
Metadata  : {'pages': 33}

información útil, oportuna, coherente, accesible, progresiva en el tiempo y técnicamente 
validada, por los organismos con competencia técnica en la materia , durante toda la 
emergencia a la comunidad y medios de comunicación , de acuerdo a las decisiones, 
prioridades y compromisos adoptados en el Comité Comunal. 
• Sistema de Evaluación de Daños y Necesidades: Establece los instrumentos necesarios para 
el levantamiento y evaluación de daños y necesidades de la comunidad afectada por una

Resultado 2
Distancia : 0.5828
Metadata  : {'pages': 33}

procesos de la Fase de Respuesta. Los flujos de comunicación entre los organismos, estructuras del 
sistema y niveles territoriales, contribuyen a una adecuada gestión de la emergencia. Por otra parte, 
la entrega de información clara, precisa, objetiva e inclusiva a la comunidad y medios de 
comunicación constituyen parte fundamental de la responsabilidad del municipio y de las 
auto

## Conclusión

### Validación de los resultados

La consulta recuperó fragmentos relacionados con la temática solicitada.

Se observa que los resultados contienen información pertinente proveniente del documento oficial indexado, lo que confirma que la búsqueda semántica sobre ChromaDB funciona correctamente.

Estos fragmentos constituyen el conjunto de candidatos que será utilizado en la siguiente etapa de reranking para seleccionar el contexto final que recibirá el modelo de lenguaje.